# M5 External Data Integration & Feature Preparation

**What this notebook does:**
1. Re-run data loading and cleaning (same pipeline as `01_data_preparation.ipynb`)  
2. Join external weather data (exploratory — see note below)  
3. Compute SNAP and event indicators  
4. Export enriched long-format table for feature engineering

> **Note:** Weather features (temperature, precipitation) were explored in this notebook but produced no meaningful RMSE improvement (< 0.0001). They are **not** included in the final model. The weather join code is retained for reproducibility.

In [1]:
import os, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

## 1. Configuration

In [2]:
# (No Colab mount needed when running locally — DATA_DIR above points to your data folder.)


In [3]:
# Update DATA_DIR to point to your local copy of the Kaggle M5 data.
# Default: place the CSV files in  <repo_root>/data/raw/
DATA_DIR = Path("../data/raw")

CALENDAR_PATH = DATA_DIR / "calendar.csv"
PRICES_PATH   = DATA_DIR / "sell_prices.csv"

if (DATA_DIR / "sales_train_evaluation.csv").exists():
    SALES_PATH = DATA_DIR / "sales_train_evaluation.csv"
else:
    SALES_PATH = DATA_DIR / "sales_train_validation.csv"

FAST_MODE    = False    # Sample N_SERIES for quick runs; set False for full data
N_SERIES     = 2000
HISTORY_DAYS = 500     # Days of history to keep in long format
VALID_DAYS   = 28
RANDOM_STATE = 1234

OUTPUT_DIR = Path("./cleaned_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Sales file:", SALES_PATH.name)
print("Fast mode:", FAST_MODE)


Sales file: sales_train_evaluation.csv
Fast mode: False


## 2. Load Raw Data

In [4]:
calendar_raw = pd.read_csv(CALENDAR_PATH)
prices_raw   = pd.read_csv(PRICES_PATH)
sales_raw    = pd.read_csv(SALES_PATH)

print("calendar:", calendar_raw.shape)
print("prices:  ", prices_raw.shape)
print("sales:   ", sales_raw.shape)

calendar: (1969, 14)
prices:   (6841121, 4)
sales:    (30490, 1947)


## 3. Clean Calendar

- Parse `date` to datetime  
- Fill event NaNs with `"NoEvent"` (1,807 of 1,969 days have no event)  
- Downcast integer columns

In [5]:
calendar = calendar_raw.copy()

# Parse date
calendar["date"] = pd.to_datetime(calendar["date"])

# Fill event NaNs — these are not missing data, they are "no event happened"
# for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
#   print(f"  {col}: filled {calendar[col].isna().sum()} NaN → 'NoEvent'")
#    calendar[col] = calendar[col].fillna("NoEvent")

# Downcast
for col in ["wday", "month", "snap_CA", "snap_TX", "snap_WI"]:
    calendar[col] = calendar[col].astype(np.int8)
calendar["year"] = calendar["year"].astype(np.int16)

# Add integer day index for efficient operations downstream
calendar["d_num"] = calendar["d"].str.extract(r"(\d+)").astype(np.int16)

print(f"\n✓ Calendar cleaned: {calendar.shape}")


✓ Calendar cleaned: (1969, 15)


## 4. Clean Prices

- Downcast `sell_price` from float64 → float32  
- Quick sanity check

In [6]:
prices = prices_raw.copy()
prices["sell_price"] = prices["sell_price"].astype(np.float32)

print(f"Price range: ${prices['sell_price'].min():.2f} – ${prices['sell_price'].max():.2f}")
print(f"NaN prices: {prices['sell_price'].isna().sum()}")
print(f"✓ Prices cleaned: {prices.shape}")

Price range: $0.01 – $107.32
NaN prices: 0
✓ Prices cleaned: (6841121, 4)


## 5. Clean Sales

- Downcast 1,941 day columns from int64 → int16 (saves ~440 MB)  
- Compute `first_sale_day` per series to identify leading zeros  
  - 77% of series have leading zeros — these are items that weren't on the shelf yet  
  - Downstream notebooks can use `is_active` to exclude pre-shelf zeros from training

In [7]:
sales = sales_raw.copy()
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
d_cols = sorted([c for c in sales.columns if c.startswith("d_")],
                key=lambda x: int(x.split("_")[1]))

# Downcast day columns
mem_before = sales[d_cols].memory_usage(deep=True).sum() / 1e6
for col in d_cols:
    sales[col] = sales[col].astype(np.int16)
mem_after = sales[d_cols].memory_usage(deep=True).sum() / 1e6
print(f"Memory: {mem_before:.0f} MB → {mem_after:.0f} MB (saved {mem_before - mem_after:.0f} MB)")

# Compute first non-zero sale day per series
mat = sales[d_cols].values
first_nonzero = np.argmax(mat > 0, axis=1)
all_zero = mat.sum(axis=1) == 0
first_nonzero[all_zero] = mat.shape[1]

series_meta = sales[id_cols].copy()
series_meta["first_sale_day"] = first_nonzero + 1  # 1-indexed to match d_1, d_2, ...
series_meta["has_leading_zeros"] = (first_nonzero > 0).astype(np.int8)

print(f"Series with leading zeros: {series_meta['has_leading_zeros'].sum()} / {len(series_meta)}")
print(f"✓ Sales cleaned: {sales.shape}")

Memory: 473 MB → 118 MB (saved 355 MB)
Series with leading zeros: 23511 / 30490
✓ Sales cleaned: (30490, 1947)


In [8]:
# Optional: sample for fast iteration
if FAST_MODE:
    sampled_ids = sales["id"].sample(n=min(N_SERIES, len(sales)), random_state=RANDOM_STATE).values
    sales = sales[sales["id"].isin(sampled_ids)].reset_index(drop=True)
    series_meta = series_meta[series_meta["id"].isin(sampled_ids)].reset_index(drop=True)
    print(f"FAST_MODE: sampled {len(sales)} series")
else:
    print(f"Full data: {len(sales)} series")

Full data: 30490 series


## 6. Melt to Long Format & Merge

- Keep only the most recent `HISTORY_DAYS + VALID_DAYS` columns  
- Melt wide → long  
- Left-join calendar and prices  
- Forward/back-fill NaN prices (items that appear mid-history have no price for early days)  
- Create `is_active` flag and state-specific `snap_flag`

In [9]:
# Keep recent days only
keep_d = d_cols[-(HISTORY_DAYS + VALID_DAYS):]
print(f"Keeping {len(keep_d)} day columns: {keep_d[0]} → {keep_d[-1]}")

# Melt
long_df = sales[id_cols + keep_d].melt(
    id_vars=id_cols, value_vars=keep_d, var_name="d", value_name="demand"
)
long_df["demand"] = long_df["demand"].astype(np.int16)

# Merge calendar
cal_merge = ["d", "d_num", "date", "wm_yr_wk", "wday", "month", "year", "weekday",
             "snap_CA", "snap_TX", "snap_WI",
             "event_name_1", "event_type_1", "event_name_2", "event_type_2"]
long_df = long_df.merge(calendar[cal_merge], on="d", how="left")
long_df["date"] = pd.to_datetime(long_df["date"])

# Merge prices
long_df = long_df.merge(
    prices[["store_id", "item_id", "wm_yr_wk", "sell_price"]],
    on=["store_id", "item_id", "wm_yr_wk"], how="left"
)

# Sort by series + time
long_df = long_df.sort_values(["id", "d_num"]).reset_index(drop=True)

print(f"Long format: {long_df.shape}")
print(f"Price NaN before fill: {long_df['sell_price'].isna().sum():,}")

Keeping 528 day columns: d_1414 → d_1941
Long format: (16098720, 23)
Price NaN before fill: 119,168


In [10]:
# Forward/back-fill price gaps within each series
long_df["sell_price"] = long_df.groupby("id")["sell_price"].transform(
    lambda s: s.ffill().bfill()
)
long_df["sell_price"] = long_df["sell_price"].fillna(0).astype(np.float32)
print(f"Price NaN after fill: {long_df['sell_price'].isna().sum()}")

# is_active: whether the item was on shelf at this date
first_sale = series_meta.set_index("id")["first_sale_day"]
long_df["is_active"] = (long_df["d_num"] >= long_df["id"].map(first_sale)).astype(np.int8)
print(f"Active rows: {long_df['is_active'].mean()*100:.1f}%")

# snap_flag: state-specific SNAP indicator
long_df["snap_flag"] = np.int8(0)
for state, col in [("CA", "snap_CA"), ("TX", "snap_TX"), ("WI", "snap_WI")]:
    mask = long_df["state_id"] == state
    long_df.loc[mask, "snap_flag"] = long_df.loc[mask, col].astype(np.int8)

print(f"\n✓ Long-format table ready: {long_df.shape}")
display(long_df.head())

Price NaN after fill: 0
Active rows: 99.2%

✓ Long-format table ready: (16098720, 25)


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,d_num,date,wm_yr_wk,wday,month,year,weekday,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,is_active,snap_flag
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1414,1,1414,2014-12-12,11445,7,12,2014,Friday,0,1,1,NaN,NaN,NaN,NaN,2.24,1,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1415,2,1415,2014-12-13,11446,1,12,2014,Saturday,0,1,0,NaN,NaN,NaN,NaN,2.24,1,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1416,2,1416,2014-12-14,11446,2,12,2014,Sunday,0,0,1,NaN,NaN,NaN,NaN,2.24,1,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1417,1,1417,2014-12-15,11446,3,12,2014,Monday,0,1,1,NaN,NaN,NaN,NaN,2.24,1,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1418,1,1418,2014-12-16,11446,4,12,2014,Tuesday,0,0,0,NaN,NaN,NaN,NaN,2.24,1,0


In [11]:
print(long_df['date'].min())
print(long_df['date'].max())

2014-12-12 00:00:00
2016-05-22 00:00:00


## 7. External Dataset Joins (Placeholder)

Uncomment any block below to join external data.  
Supply the matching CSV file in your `DATA_DIR`.

### Weather Data

In [ ]:
weather = pd.read_csv(DATA_DIR / "weather.csv")
weather["DATE"] = pd.to_datetime(weather["DATE"])

# keep only recent days
weather = weather[
    (weather["DATE"] >= "2014-12-11") &
    (weather["DATE"] <= "2016-05-23")
]

# create state column
station_to_state = {
    "USW00023174": "CA",  # Los Angeles
    "USW00003927": "TX",  # Dallas
    "USW00014839": "WI"   # Milwaukee
}

weather["state"] = weather["STATION"].map(station_to_state)

weather["SNOW"] = weather["SNOW"].fillna(0).astype(np.float32)

# keep only useful columns
weather = weather[["state", "DATE", "PRCP", "SNOW", "TMIN", "TMAX"]]

# convert all column names to lowercase
weather.columns = weather.columns.str.lower()
weather.head()

,state,date,tmin,tmax
1412,CA,2014-12-11,60,65
1413,CA,2014-12-12,54,62
1414,CA,2014-12-13,50,62
1415,CA,2014-12-14,48,63
1416,CA,2014-12-15,49,64


In [13]:
# add extreme weather flags

weather["month"] = weather["date"].dt.month

# TMAX thresholds (per state + month)
tmax_hot_thresh = weather.groupby(["state", "month"])["tmax"].transform(
    lambda x: x.quantile(0.9)
)

tmax_cold_thresh = weather.groupby(["state", "month"])["tmax"].transform(
    lambda x: x.quantile(0.1)
)

# TMIN thresholds
tmin_cold_thresh = weather.groupby(["state", "month"])["tmin"].transform(
    lambda x: x.quantile(0.1)
)

tmin_warm_thresh = weather.groupby(["state", "month"])["tmin"].transform(
    lambda x: x.quantile(0.9)
)

# create flags
weather["is_hot_day"] = (weather["tmax"] >= tmax_hot_thresh).astype(int)
weather["is_cold_night"] = (weather["tmin"] <= tmin_cold_thresh).astype(int)

# state + month normalized temperature z-scores
weather["tmax_z"] = (
    weather["tmax"] - weather.groupby(["state", "month"])["tmax"].transform("mean")
) / weather.groupby(["state", "month"])["tmax"].transform("std")

weather["tmin_z"] = (
    weather["tmin"] - weather.groupby(["state", "month"])["tmin"].transform("mean")
) / weather.groupby(["state", "month"])["tmin"].transform("std")

# optional: handle groups with std = 0
weather["tmax_z"] = weather["tmax_z"].replace([np.inf, -np.inf], np.nan).fillna(0)
weather["tmin_z"] = weather["tmin_z"].replace([np.inf, -np.inf], np.nan).fillna(0)

weather.head()

,state,date,tmin,tmax,month,is_hot_day,is_cold_night,tmax_z,tmin_z
1412,CA,2014-12-11,60,65,12,0,0,0.015532,2.122359
1413,CA,2014-12-12,54,62,12,0,0,-0.469071,0.982641
1414,CA,2014-12-13,50,62,12,0,0,-0.469071,0.222829
1415,CA,2014-12-14,48,63,12,0,0,-0.307537,-0.157076
1416,CA,2014-12-15,49,64,12,0,0,-0.146002,0.032876


In [14]:
# % of extreme weather
weather[["is_hot_day", "is_cold_night"]].mean()

is_hot_day       0.127044
is_cold_night    0.135220
dtype: float64

In [ ]:
# add snow and rain flags
weather["is_snowy"] = (weather["SNOW"] > 0).astype(int)
weather["is_rainy"] = (weather["PRCP"] > 0).astype(int)

In [15]:
# ==============================================================================
# EXTERNAL DATASETS — UNCOMMENT AND ADAPT AS NEEDED
# ==============================================================================

# ---------- Unemployment by state (monthly, from BLS) ----------
# unemployment = pd.read_csv(DATA_DIR / "state_unemployment.csv")
# unemployment["date"] = pd.to_datetime(unemployment["date"])
# unemployment["ym"] = unemployment["date"].dt.to_period("M")
# long_df["ym"] = long_df["date"].dt.to_period("M")
# long_df = long_df.merge(unemployment[["ym", "state_id", "unemployment_rate"]],
#                         on=["ym", "state_id"], how="left")
# long_df.drop(columns=["ym"], inplace=True)


# ---------- Daily weather by state (from NOAA / Visual Crossing) ----------
long_df = long_df.merge(
    weather,
    left_on=["date", "state_id"],
    right_on=["date", "state"],
    how="left"
)
long_df.head()


# ---------- CPI / inflation (monthly, from BLS) ----------
# cpi = pd.read_csv(DATA_DIR / "us_cpi_monthly.csv")
# cpi["date"] = pd.to_datetime(cpi["date"])
# cpi["ym"] = cpi["date"].dt.to_period("M")
# long_df["ym"] = long_df["date"].dt.to_period("M")
# long_df = long_df.merge(cpi[["ym", "cpi_index"]], on="ym", how="left")
# long_df["real_price"] = long_df["sell_price"] / long_df["cpi_index"] * 100
# long_df.drop(columns=["ym"], inplace=True)


# ---------- Google Trends by product category ----------
# trends = pd.read_csv(DATA_DIR / "google_trends_by_category.csv")
# trends["date"] = pd.to_datetime(trends["date"])
# long_df = long_df.merge(trends[["date", "cat_id", "search_interest"]],
#                         on=["date", "cat_id"], how="left")


# ---------- State-level holidays (via `pip install holidays`) ----------
# import holidays
# for st in ["CA", "TX", "WI"]:
#     hols = holidays.US(years=range(2011, 2017), state=st)
#     calendar[f"holiday_{st}"] = calendar["date"].dt.date.isin(hols).astype(np.int8)
# # Then re-merge the new columns into long_df

,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,d_num,date,wm_yr_wk,wday,month_x,year,weekday,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,is_active,snap_flag,state,tmin,tmax,month_y,is_hot_day,is_cold_night,tmax_z,tmin_z
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1414,1,1414,2014-12-12,11445,7,12,2014,Friday,0,1,1,NaN,NaN,NaN,NaN,2.24,1,0,CA,54,62,12,0,0,-0.469071,0.982641
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1415,2,1415,2014-12-13,11446,1,12,2014,Saturday,0,1,0,NaN,NaN,NaN,NaN,2.24,1,0,CA,50,62,12,0,0,-0.469071,0.222829
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1416,2,1416,2014-12-14,11446,2,12,2014,Sunday,0,0,1,NaN,NaN,NaN,NaN,2.24,1,0,CA,48,63,12,0,0,-0.307537,-0.157076
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1417,1,1417,2014-12-15,11446,3,12,2014,Monday,0,1,1,NaN,NaN,NaN,NaN,2.24,1,0,CA,49,64,12,0,0,-0.146002,0.032876
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1418,1,1418,2014-12-16,11446,4,12,2014,Tuesday,0,0,0,NaN,NaN,NaN,NaN,2.24,1,0,CA,53,60,12,0,0,-0.792140,0.792688


### Event Feature Engineering + Event Proximity

In [16]:
# Commented out the "Fill event NaNs" from Clean Calendar -- keeping the NAs

calendar["is_event"] = (
    calendar["event_name_1"].notna() |
    calendar["event_name_2"].notna()
).astype(int)

# sorted unique event dates from FULL calendar
event_dates = pd.Series(
    sorted(calendar.loc[calendar["is_event"] == 1, "date"].drop_duplicates())
).reset_index(drop=True)

dates = calendar["date"]

# index of next event
next_idx = np.searchsorted(event_dates.values, dates.values, side="left")
prev_idx = next_idx - 1

# previous event date
prev_event = pd.Series(pd.NaT, index=calendar.index, dtype="datetime64[ns]")
valid_prev = prev_idx >= 0
prev_event.loc[valid_prev] = event_dates.iloc[prev_idx[valid_prev]].values

# next event date
next_event = pd.Series(pd.NaT, index=calendar.index, dtype="datetime64[ns]")
valid_next = next_idx < len(event_dates)
next_event.loc[valid_next] = event_dates.iloc[next_idx[valid_next]].values

# distances
calendar["days_since_event"] = (dates - prev_event).dt.days
calendar["days_until_event"] = (next_event - dates).dt.days

# optional: exact event day = 0 (commented out because there were two consecutive event days)
# calendar.loc[calendar["is_event"] == 1, ["days_since_event", "days_until_event"]] = 0

In [17]:
event_features = calendar[[
    "date", "is_event", "days_since_event", "days_until_event"
]].drop_duplicates()

long_df = long_df.drop(columns=[
    c for c in ["is_event", "days_since_event", "days_until_event"]
    if c in long_df.columns
])

long_df = long_df.merge(event_features, on="date", how="left")

In [19]:
# --- event type flags (auto from unique values) ---
event_types = pd.concat([
    long_df["event_type_1"],
    long_df["event_type_2"]
]).dropna().unique()

for et in event_types:
    col_name = f"is_{et.lower()}_event"
    long_df[col_name] = (
        (long_df["event_type_1"] == et) |
        (long_df["event_type_2"] == et)
    ).astype(int)

long_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,d_num,date,wm_yr_wk,wday,month_x,year,weekday,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,is_active,snap_flag,state,tmin,tmax,month_y,is_hot_day,is_cold_night,tmax_z,tmin_z,is_event,days_since_event,days_until_event,is_religious_event,is_national_event,is_sporting_event,is_cultural_event
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1414,1,1414,2014-12-12,11445,7,12,2014,Friday,0,1,1,NaN,NaN,NaN,NaN,2.24,1,0,CA,54,62,12,0,0,-0.469071,0.982641,0,15.0,12,0,0,0,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1415,2,1415,2014-12-13,11446,1,12,2014,Saturday,0,1,0,NaN,NaN,NaN,NaN,2.24,1,0,CA,50,62,12,0,0,-0.469071,0.222829,0,16.0,11,0,0,0,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1416,2,1416,2014-12-14,11446,2,12,2014,Sunday,0,0,1,NaN,NaN,NaN,NaN,2.24,1,0,CA,48,63,12,0,0,-0.307537,-0.157076,0,17.0,10,0,0,0,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1417,1,1417,2014-12-15,11446,3,12,2014,Monday,0,1,1,NaN,NaN,NaN,NaN,2.24,1,0,CA,49,64,12,0,0,-0.146002,0.032876,0,18.0,9,0,0,0,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1418,1,1418,2014-12-16,11446,4,12,2014,Tuesday,0,0,0,NaN,NaN,NaN,NaN,2.24,1,0,CA,53,60,12,0,0,-0.792140,0.792688,0,19.0,8,0,0,0,0


In [21]:
# verify
long_df[long_df["date"] == '2014-12-23']

,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,d_num,date,wm_yr_wk,wday,month_x,year,weekday,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,is_active,snap_flag,state,tmin,tmax,month_y,is_hot_day,is_cold_night,tmax_z,tmin_z,is_event,days_since_event,days_until_event,is_religious_event,is_national_event,is_sporting_event,is_cultural_event
11,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1425,1,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,2.24,1,0,CA,52,79,12,1,0,2.277014,0.602735,0,26.0,1,0,0,0,0
539,FOODS_1_001_CA_2_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_2,CA,d_1425,0,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,2.24,1,0,CA,52,79,12,1,0,2.277014,0.602735,0,26.0,1,0,0,0,0
1067,FOODS_1_001_CA_3_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,d_1425,1,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,2.24,1,0,CA,52,79,12,1,0,2.277014,0.602735,0,26.0,1,0,0,0,0
1595,FOODS_1_001_CA_4_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_4,CA,d_1425,0,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,2.24,1,0,CA,52,79,12,1,0,2.277014,0.602735,0,26.0,1,0,0,0,0
2123,FOODS_1_001_TX_1_evaluation,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,d_1425,0,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,2.24,1,0,TX,44,53,12,0,0,-0.770477,0.145023,0,26.0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16096091,HOUSEHOLD_2_516_TX_2_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,TX_2,TX,d_1425,0,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,5.94,1,0,TX,44,53,12,0,0,-0.770477,0.145023,0,26.0,1,0,0,0,0
16096619,HOUSEHOLD_2_516_TX_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,TX_3,TX,d_1425,0,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,5.94,1,0,TX,44,53,12,0,0,-0.770477,0.145023,0,26.0,1,0,0,0,0
16097147,HOUSEHOLD_2_516_WI_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_1,WI,d_1425,1,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,5.94,1,0,WI,37,46,12,0,0,0.591042,0.749336,0,26.0,1,0,0,0,0
16097675,HOUSEHOLD_2_516_WI_2_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_2,WI,d_1425,0,1425,2014-12-23,11447,4,12,2014,Tuesday,0,0,0,nan,nan,nan,nan,5.94,1,0,WI,37,46,12,0,0,0.591042,0.749336,0,26.0,1,0,0,0,0


## 8. Export

Outputs:
- `long_df_clean.parquet` — cleaned long-format table  
- `series_meta.parquet` — per-series metadata (first_sale_day, leading zeros)  

Downstream notebooks load these instead of re-running the raw pipeline.

In [20]:
# Convert category types to string for Parquet compatibility
for col in ["item_id", "dept_id", "cat_id", "store_id", "state_id", "weekday",
            "event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    if col in long_df.columns:
        long_df[col] = long_df[col].astype(str)
    if col in series_meta.columns:
        series_meta[col] = series_meta[col].astype(str)

# Export as Parquet (fast + compressed). Falls back to CSV if pyarrow is missing.
try:
    long_df.to_parquet(OUTPUT_DIR / "long_df_clean.parquet", index=False)
    series_meta.to_parquet(OUTPUT_DIR / "series_meta.parquet", index=False)
    fmt = "parquet"
except ImportError:
    print("pyarrow not found — falling back to CSV (install pyarrow for faster I/O)")
    long_df.to_csv(OUTPUT_DIR / "long_df_clean.csv", index=False)
    series_meta.to_csv(OUTPUT_DIR / "series_meta.csv", index=False)
    fmt = "csv"

for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

print(f"\n--- Summary ---")
print(f"Format: {fmt}")
print(f"Series: {long_df['id'].nunique():,}")
print(f"Rows: {len(long_df):,}")
print(f"Date range: {long_df['date'].min().date()} → {long_df['date'].max().date()}")
print(f"Active rows: {long_df['is_active'].mean()*100:.1f}%")
print(f"Zero demand: {(long_df['demand']==0).mean()*100:.1f}%")

  long_df_clean.parquet  (32.2 MB)
  series_meta.parquet  (0.3 MB)

--- Summary ---
Format: parquet
Series: 30,490
Rows: 16,098,720
Date range: 2014-12-12 → 2016-05-22
Active rows: 99.2%
Zero demand: 60.3%


## How to Load in Modeling Notebooks

```python
from pathlib import Path
import pandas as pd

CLEAN_DIR = Path("./cleaned_data")

# Parquet (if pyarrow is installed)
long_df     = pd.read_parquet(CLEAN_DIR / "long_df_clean.parquet")
series_meta = pd.read_parquet(CLEAN_DIR / "series_meta.parquet")

# Or CSV fallback
# long_df     = pd.read_csv(CLEAN_DIR / "long_df_clean.csv", parse_dates=["date"])
# series_meta = pd.read_csv(CLEAN_DIR / "series_meta.csv")

# Optional: exclude pre-shelf zeros
long_df = long_df[long_df["is_active"] == 1].reset_index(drop=True)
```